# Early FTUE analysis

## Key Findings (2026-06-26)

**Retention:**
- Different stories for each platform:
    - IOS performing equally up to D7 and showing improvement signs from D14 onwards
    - Android underperforms in most D3 and D7 with statistically significant differences


**Player Level Progression:**
- Progression speed is again different on each platform
- IOS
    - Equal max level average progression
    - Max level percentiles start showing signs of improvement

- Android
    - Average progression is -1% slower
    - High percentiles also show a bit of improvement on later DX


**Overall**
- Encouraging signs from IOS on retention D14+. Max level percentiles show signs of improvement on both platforms
- Android underperformance is not too drastic and probably related to players mix being different on new period. 
- Get more data to see D21 with more cohorts and D30 metrics

**Next**
- Add sessions and time played metrics
- Look at monetisation and economy metrics

In [1]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

<!-- hide --> 
## Aux functions

In [2]:
# hide-output
# Helper functions for weighted progression, percentile calculation, level visualisations, and retention significance
from aux_functions import (
    compute_weighted_progression,
    weighted_quantiles,
    add_event_annotations,
    add_median_lines,
    plot_percentile_comparison,
    compute_retention_significance,
    plot_retention_significance,
)

## Get data

### Player level and game day

In [3]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

new_ftue_date = dt.datetime(2026, 6, 1)
#days_from_start = (dt.datetime.today() - new_ftue_date).days
days_from_start = 28  # Set the number of days for the first window
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
#end_date2 = dt.datetime.today()-dt.timedelta(days=1)
end_date2 = new_ftue_date + dt.timedelta(days=days_from_start-1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-05-04
End Date 1: 2026-05-31
Start Date 2: 2026-06-01
End Date 2: 2026-06-28


In [4]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = False

In [5]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 28.99 GB when run.
Estimated query cost: $0.19


In [6]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [7]:
# hide-output
# Preview raw player level and game day data
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
0,B334FB70C35CE8F4,2026-05-31,2026-05-30,US,2026-05-24,2026-05-01,1,4,2,IOS,Non-Attributed,Non-Attributed,0.75.0
1,A44BBC73F3CF6844,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0
2,E1D7167E9F2473AD,2026-06-28,2026-06-28,GB,2026-06-28,2026-06-01,0,2,1,AND,Non-Attributed,Non-Attributed,0.78.0
3,9889039D9BF382F9,2026-05-30,2026-05-29,BR,2026-05-24,2026-05-01,1,5,3,AND,Non-Attributed,Non-Attributed,0.75.0
4,46A55DC95748A1C5,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,FACEBOOK,UA,0.78.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114959,A748C9D193253E5D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,IOS,Non-Attributed,Non-Attributed,0.78.0
114960,414EBFF581C6E722,2026-06-28,2026-06-27,BO,2026-06-21,2026-06-01,1,4,2,IOS,FACEBOOK,UA,0.78.0
114961,9BAD636C88E1D2C,2026-06-27,2026-06-27,CZ,2026-06-21,2026-06-01,0,2,1,IOS,FACEBOOK,UA,0.78.0
114962,E3B378842607FC4E,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0


<!-- hide --> 
### Retention

In [8]:
# hide-output
# Estimate query cost for per-install-date retention SQL
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.19 GB when run.
Estimated query cost: $0.01


In [9]:
# hide-output
# Fetch per-install-date retention data from BigQuery or load from local pickle cache
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [10]:
# hide-output
# Sort retention data and spot-check Android rows
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
1,2026-05-04,0,AND,389,389,1.000000
2,2026-05-04,1,AND,389,101,0.259640
5,2026-05-04,3,AND,389,71,0.182519
6,2026-05-04,7,AND,389,60,0.154242
8,2026-05-04,14,AND,389,38,0.097686
...,...,...,...,...,...,...
479,2026-06-26,0,AND,194,194,1.000000
480,2026-06-26,1,AND,194,51,0.262887
483,2026-06-27,0,AND,212,212,1.000000
484,2026-06-27,1,AND,212,66,0.311321


In [11]:
# hide-output
# Estimate query cost for FTUE-split retention SQL (all users)
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.19 GB when run.
Estimated query cost: $0.01


In [12]:
# hide-output
# Fetch FTUE-split retention (all users) from BigQuery or load from local pickle cache
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [13]:
# hide-output
# Sort FTUE retention data and spot-check at D14
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total[retention_data_total['dx'] == 14]

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
16,14,AND,A.Pre-FTUE revamp,14,3664,403,0.109989
17,14,AND,B.Post-FTUE revamp,14,4166,376,0.090254
19,14,IOS,A.Pre-FTUE revamp,14,8904,788,0.088500
18,14,IOS,B.Post-FTUE revamp,14,7918,819,0.103435


In [14]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.19 GB when run.
Estimated query cost: $0.01


In [15]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [16]:
# hide-output
# Sort and preview organic-only FTUE retention data
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
0,0,AND,A.Pre-FTUE revamp,28,6833,6833,1.000000
1,0,AND,B.Post-FTUE revamp,28,6601,6601,1.000000
3,0,IOS,A.Pre-FTUE revamp,28,10710,10710,1.000000
2,0,IOS,B.Post-FTUE revamp,28,13758,13758,1.000000
4,1,AND,A.Pre-FTUE revamp,27,5416,1758,0.324594
5,1,AND,B.Post-FTUE revamp,27,6326,1892,0.299083
6,1,IOS,A.Pre-FTUE revamp,27,10572,3558,0.336549
7,1,IOS,B.Post-FTUE revamp,27,13237,4289,0.324016
9,3,AND,A.Pre-FTUE revamp,25,5216,1051,0.201495
8,3,AND,B.Post-FTUE revamp,25,5964,1002,0.168008


### AB metrics

In [17]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/abmetrics.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),  
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 3.97 GB when run.
Estimated query cost: $0.03


In [18]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
abmetrics = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    abmetrics = bqc.get(query='./sql/abmetrics.sql', is_path=True, query_parameters=parameters)
    abmetrics.to_pickle('./data/abmetrics.pkl')
else:
    # Load from local cache to avoid repeated query costs
    abmetrics = pd.read_pickle('./data/abmetrics.pkl')

In [19]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,AB7310966805C1C0,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
1,72490447CC189152,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
2,E0B8F10F99A5638,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
3,A3155297937BA0BB,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
4,D4AF822BA0DE6772,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273169,DC5CC073BE9FB4E8,2026-06-21,2026-06-25,4,18,14,1,5,1,3,...,0,0,0,0,0,8,2,1,0,2026-07-03 13:45:26.013956+00:00
273170,ED94AF99015849F5,2026-06-14,2026-06-25,11,21,17,1,7,6,3,...,0,5,0,0,0,3,2,1,0,2026-07-03 13:45:26.013956+00:00
273171,9C2B1612438A11DB,2026-06-23,2026-06-25,2,17,12,1,3,1,9,...,0,5,0,0,0,6,2,1,0,2026-07-03 13:45:26.013956+00:00
273172,BAC78337CCD45243,2026-06-20,2026-06-25,5,13,9,1,5,1,5,...,0,2,0,0,0,6,0,1,0,2026-07-03 13:45:26.013956+00:00


<!-- hide --> 
## Process data

In [20]:
# hide-output
# Assign FTUE flag, cap days_since_install to match B.new window, and drop immature cohort rows
dt_mode = 'install_dt'
processed_data = data.copy()

processed_data['install_dt'] = processed_data[dt_mode]

processed_data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in processed_data['acquisition_type']]
processed_data['FTUE_flag'] = ['B.new' if x >='0.76.0' else 'A.old' for x in processed_data['install_build_version']]

# Making comparison fair
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime('2026-06-01')).days
processed_data = processed_data[~(processed_data['days_since_install'] > max_dayx_B_new)]

processed_data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
processed_data = processed_data[processed_data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(processed_data['install_dt'])).dt.days - min_days_since_install]


# Add a country filter just to see if metrics align better with the US-only data. This is a temporary filter for testing purposes.
processed_data = processed_data[processed_data['country_code'] == 'US']

processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,B334FB70C35CE8F4,2026-05-31,2026-05-30,US,2026-05-24,2026-05-01,1,4,2,IOS,Non-Attributed,Non-Attributed,0.75.0,N,A.old,dummy
1,A44BBC73F3CF6844,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
4,46A55DC95748A1C5,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,FACEBOOK,UA,0.78.0,N,B.new,dummy
7,A3FCB6D84BFC30F9,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
8,921AF1EAC26A2E86,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,4,2,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114949,1ED7E281DEA38E1D,2026-06-28,2026-06-27,US,2026-06-21,2026-06-01,1,6,4,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114950,6EA95751E0985B67,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114957,B7CE59BBB4645B3D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114959,A748C9D193253E5D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy


In [21]:
# hide-output
# Sanity-check unique user counts per FTUE group
test = processed_data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,3887
1,B.new,4999


## Retention

In [22]:
# hide-output
# Add combined dx_platform column for the per-install-date retention line chart
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data

,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
1,2026-05-04,0,AND,389,389,1.000000,0_AND
0,2026-05-04,0,IOS,589,589,1.000000,0_IOS
2,2026-05-04,1,AND,389,101,0.259640,1_AND
3,2026-05-04,1,IOS,589,188,0.319185,1_IOS
5,2026-05-04,3,AND,389,71,0.182519,3_AND
...,...,...,...,...,...,...,...
482,2026-06-27,0,IOS,513,513,1.000000,0_IOS
484,2026-06-27,1,AND,212,66,0.311321,1_AND
485,2026-06-27,1,IOS,513,155,0.302144,1_IOS
486,2026-06-28,0,AND,217,217,1.000000,0_AND


In [23]:
# Per-install-date retention rate over time by dx/platform (figure disabled — used for investigation only)
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [24]:
# hide-output
# Preview FTUE-split retention totals (all users)
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,28,8217,8217,1.000000
0,0,AND,B.Post-FTUE revamp,28,7511,7511,1.000000
2,0,IOS,A.Pre-FTUE revamp,28,13813,13813,1.000000
3,0,IOS,B.Post-FTUE revamp,28,17236,17236,1.000000
4,1,AND,A.Pre-FTUE revamp,27,6563,1905,0.290264
5,1,AND,B.Post-FTUE revamp,27,7165,2000,0.279135
7,1,IOS,A.Pre-FTUE revamp,27,13664,4363,0.319306
6,1,IOS,B.Post-FTUE revamp,27,16610,5323,0.320470
8,3,AND,A.Pre-FTUE revamp,25,6255,1160,0.185452
9,3,AND,B.Post-FTUE revamp,25,6788,1075,0.158368


### Cohort sizes

In [25]:
# Bar chart: cohort sizes at each retention checkpoint by FTUE group and platform
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate 

In [26]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (all users)
# Each Dx is tested independently with a two-proportion z-test; Wilson 95% CIs shown as error bars
dummy = plot_retention_significance(
    retention_data_total,
    title='Retention rate by FTUE group',
)

### Retention rate (Organics only)

In [27]:
# hide-output
# Bar chart: D1–D21 retention rates by FTUE group and platform (organic / non-attributed only)
# Each Dx is tested independently; small D21 organic cohorts annotated with users needed for significance
dummy = plot_retention_significance(
    retention_data_total_na,
    title='Retention rate by FTUE group — organic only',
)

## Player max level distribution

In [28]:
# hide-output
# Preview player data
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,B334FB70C35CE8F4,2026-05-31,2026-05-30,US,2026-05-24,2026-05-01,1,4,2,IOS,Non-Attributed,Non-Attributed,0.75.0,N,A.old,dummy
1,A44BBC73F3CF6844,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
4,46A55DC95748A1C5,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,FACEBOOK,UA,0.78.0,N,B.new,dummy
7,A3FCB6D84BFC30F9,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
8,921AF1EAC26A2E86,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,4,2,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114949,1ED7E281DEA38E1D,2026-06-28,2026-06-27,US,2026-06-21,2026-06-01,1,6,4,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114950,6EA95751E0985B67,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114957,B7CE59BBB4645B3D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114959,A748C9D193253E5D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy


In [50]:
# hide-output
# Build level funnel with P10/P50/P90 percentiles per FTUE group, platform, and day since install
days_since_install_limit = 21

pl_ftue_funnel_agg = processed_data.groupby(['max_level','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_level','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = processed_data.groupby(['days_since_install', 'max_level', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist[level_dist.days_since_install<=days_since_install_limit].groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_level', include_groups=False
).reset_index()

# Rename columns for clarity
level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_level', 'p50': 'p50_max_level', 'p90': 'p90_max_level'})

# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21,28])]


pl_ftue_funnel_agg

,max_level,FTUE_flag,platform,days_since_install,users,total_users,pctg_users,pctg_diff_users,combined_dimension,p10_max_level,p50_max_level,p90_max_level
0,1,A.old,AND,0,169,1098,0.153916,0.0,A.old | 0,1.0,3.0,5.0
1,1,A.old,AND,1,17,504,0.033730,0.0,A.old | 1,3.0,5.0,7.0
3,1,A.old,AND,3,6,336,0.017857,0.0,A.old | 3,4.0,7.0,11.0
12,1,A.old,AND,14,1,129,0.007752,0.0,A.old | 14,6.0,12.0,19.0
15,1,A.old,IOS,0,367,2750,0.133455,0.0,A.old | 0,1.0,3.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2623,154,B.new,AND,7,1,225,0.004444,0.0,B.new | 7,5.0,9.0,14.0
2630,154,B.new,AND,14,1,122,0.008197,0.0,B.new | 14,6.0,13.0,21.0
2635,155,B.new,AND,3,1,314,0.003185,0.0,B.new | 3,4.0,7.0,11.0
2640,155,B.new,AND,14,2,122,0.016393,0.0,B.new | 14,6.0,13.0,21.0


In [51]:
# hide-output
# Define level milestone annotations for A.old and B.new FTUE feature unlock points
events_config = {
    #'A.old': [
    #    {'level': 7, 'name': 'SP', 'color':'blue'},
    #    {'level': 8, 'name': 'Deco', 'color': 'blue'},
    #    {'level': 10, 'name': 'TimedC', 'color': 'blue'},
    #    {'level': 16, 'name': 'TA', 'color': 'blue'},
    #    {'level': 20, 'name': 'GenB', 'color': 'blue'},
    #   {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    #],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}

In [52]:
max_level = 20

# Level distribution charts: user counts, percentage share, and day-over-day diff (up to level 30)
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= max_level], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='users',
              color='combined_dimension',
              title='Players level distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_level', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= max_level], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each level (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=True)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= max_level], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each level (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile max level reached

In [53]:
# Band chart: P10/P50/P90 level progression comparison between A.old and B.new
plot_percentile_comparison(level_pcts_by_group, percentile='all')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Average max level reached

In [54]:
# hide-output
# Compute weighted average max level by FTUE group, platform, and days since install
days_since_install_baseline = 28

data_filtered = processed_data[processed_data['days_since_install'] <= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=10)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_level'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_level'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg

,dummy,days_since_install,FTUE_flag,platform,cohort_users,weighted_avg_max_level,combined_dimension,pctg_diff_max_level,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,AND,1085,3.240437,dummy | A.old,0.000000,1085,1.000000,0.000000
1,dummy,0,A.old,IOS,2727,3.384727,dummy | A.old,0.000000,2727,1.000000,0.000000
2,dummy,0,B.new,AND,1081,3.233364,dummy | B.new,-0.002183,1081,1.000000,0.000000
3,dummy,0,B.new,IOS,3847,3.311543,dummy | B.new,-0.021622,3847,1.000000,0.000000
4,dummy,1,A.old,AND,490,5.609127,dummy | A.old,0.000000,1085,0.451613,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
76,dummy,20,B.new,IOS,125,15.721393,dummy | B.new,-0.161266,3847,0.032493,0.969067
77,dummy,21,A.old,IOS,44,18.465649,dummy | A.old,0.000000,2727,0.016135,0.000000
78,dummy,21,B.new,IOS,54,16.732919,dummy | B.new,-0.093835,3847,0.014037,-0.130030
79,dummy,22,B.new,IOS,22,16.818841,dummy | B.new,0.000000,3847,0.005719,0.000000


In [55]:
# hide-output
# Bar chart: surviving cohort size at each day since install by FTUE group
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [56]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [57]:
# hide-output
# Sort data by user and day for per-user progression analysis
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
217,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0
63569,1007A1182B8808,2026-06-03,2026-06-03,BR,2026-05-31,2026-06-01,0,4,2,AND,FACEBOOK,Creative Test,0.76.0
63388,1007A1182B8808,2026-06-04,2026-06-03,BR,2026-05-31,2026-06-01,1,5,3,AND,FACEBOOK,Creative Test,0.76.0
14401,1007FD05E4DA180,2026-05-07,2026-05-07,FJ,2026-05-03,2026-05-01,0,1,1,IOS,FACEBOOK,UA,0.74.0
100336,1008C486C3810433,2026-06-18,2026-06-18,PH,2026-06-14,2026-06-01,0,6,4,IOS,Non-Attributed,Non-Attributed,0.77.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36743,FFFE079DACB6000A,2026-05-19,2026-05-16,US,2026-05-10,2026-05-01,3,8,5,IOS,LIFTOFF,UA,0.75.0
37869,FFFE079DACB6000A,2026-05-20,2026-05-16,US,2026-05-10,2026-05-01,4,8,5,IOS,LIFTOFF,UA,0.75.0
37223,FFFE079DACB6000A,2026-05-22,2026-05-16,US,2026-05-10,2026-05-01,6,8,5,IOS,LIFTOFF,UA,0.75.0
38572,FFFE079DACB6000A,2026-05-23,2026-05-16,US,2026-05-10,2026-05-01,7,9,6,IOS,LIFTOFF,UA,0.75.0


## Engagement metrics

In [58]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,AB7310966805C1C0,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
1,72490447CC189152,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
2,E0B8F10F99A5638,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
3,A3155297937BA0BB,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
4,D4AF822BA0DE6772,2026-06-08,2026-06-08,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-06-18 18:29:41.321007+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273169,DC5CC073BE9FB4E8,2026-06-21,2026-06-25,4,18,14,1,5,1,3,...,0,0,0,0,0,8,2,1,0,2026-07-03 13:45:26.013956+00:00
273170,ED94AF99015849F5,2026-06-14,2026-06-25,11,21,17,1,7,6,3,...,0,5,0,0,0,3,2,1,0,2026-07-03 13:45:26.013956+00:00
273171,9C2B1612438A11DB,2026-06-23,2026-06-25,2,17,12,1,3,1,9,...,0,5,0,0,0,6,2,1,0,2026-07-03 13:45:26.013956+00:00
273172,BAC78337CCD45243,2026-06-20,2026-06-25,5,13,9,1,5,1,5,...,0,2,0,0,0,6,0,1,0,2026-07-03 13:45:26.013956+00:00


In [59]:
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,B334FB70C35CE8F4,2026-05-31,2026-05-30,US,2026-05-24,2026-05-01,1,4,2,IOS,Non-Attributed,Non-Attributed,0.75.0,N,A.old,dummy
1,A44BBC73F3CF6844,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
4,46A55DC95748A1C5,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,FACEBOOK,UA,0.78.0,N,B.new,dummy
7,A3FCB6D84BFC30F9,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
8,921AF1EAC26A2E86,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,4,2,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114949,1ED7E281DEA38E1D,2026-06-28,2026-06-27,US,2026-06-21,2026-06-01,1,6,4,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114950,6EA95751E0985B67,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114957,B7CE59BBB4645B3D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy
114959,A748C9D193253E5D,2026-06-27,2026-06-27,US,2026-06-21,2026-06-01,0,1,1,IOS,Non-Attributed,Non-Attributed,0.78.0,N,B.new,dummy


In [60]:
days_since_install_limit = 21

engagement_data = abmetrics[['user_id','dt','days_since_install','n_sessions','n_mins_in_game','n_merges','n_tasks_completed']]
engagement_data = processed_data[['user_id','dt','FTUE_flag','platform']].merge(engagement_data, on=['user_id', 'dt'], how='left')
engagement_data = engagement_data[engagement_data['days_since_install'] <= days_since_install_limit]
engagement_data

,user_id,dt,FTUE_flag,platform,days_since_install,n_sessions,n_mins_in_game,n_merges,n_tasks_completed
0,B334FB70C35CE8F4,2026-05-31,A.old,IOS,1,2,8,97.0,3
1,A44BBC73F3CF6844,2026-06-28,B.new,IOS,0,3,57,643.0,16
2,46A55DC95748A1C5,2026-06-28,B.new,IOS,0,3,25,539.0,14
3,A3FCB6D84BFC30F9,2026-06-28,B.new,IOS,0,1,11,134.0,3
4,921AF1EAC26A2E86,2026-06-28,B.new,AND,0,1,21,426.0,9
...,...,...,...,...,...,...,...,...,...
40412,1ED7E281DEA38E1D,2026-06-28,B.new,IOS,1,5,26,397.0,7
40413,6EA95751E0985B67,2026-06-27,B.new,IOS,0,1,24,282.0,8
40414,B7CE59BBB4645B3D,2026-06-27,B.new,AND,0,1,6,114.0,1
40415,A748C9D193253E5D,2026-06-27,B.new,IOS,0,3,3,5.0,0


In [61]:
# calculate percentiles P10, P50 and P90 for n_sessions, n_mins_in_game, and n_merges by FTUE_flag, platform, and days_since_install
percentiles = [0.1, 0.5, 0.9]

# Calculate P10, P50, P90 percentiles for n_sessions from engagement_data
percentiles_sessions = engagement_data.groupby(['FTUE_flag', 'platform', 'days_since_install'])['n_sessions'].quantile(percentiles).unstack(fill_value=0).reset_index()
percentiles_sessions.columns = ['FTUE_flag', 'platform', 'days_since_install', 'p10_sessions', 'p50_sessions', 'p90_sessions']
percentiles_sessions


engagement_data_agg = engagement_data.groupby(['FTUE_flag','platform','days_since_install']).agg(
    avg_sessions=('n_sessions', 'mean'),
    avg_n_mins_in_game=('n_mins_in_game', 'mean'),
    avg_n_merges=('n_merges', 'mean'),
    avg_n_tasks_completed=('n_tasks_completed', 'mean'),
    total_users=('user_id', 'nunique')
).reset_index()

engagement_data_agg = engagement_data_agg.merge(percentiles_sessions, on=['FTUE_flag', 'platform', 'days_since_install'], how='left') 

engagement_data_agg.sort_values(['days_since_install','platform','FTUE_flag'], inplace=True)
engagement_data_agg['pctg_diff_avg_sessions'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_sessions'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_mins_in_game'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_mins_in_game'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_merges'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_merges'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_tasks_completed'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_tasks_completed'].pct_change().fillna(0)

engagement_data_agg

,FTUE_flag,platform,days_since_install,avg_sessions,avg_n_mins_in_game,avg_n_merges,avg_n_tasks_completed,total_users,p10_sessions,p50_sessions,p90_sessions,pctg_diff_avg_sessions,pctg_diff_avg_n_mins_in_game,pctg_diff_avg_n_merges,pctg_diff_avg_n_tasks_completed
0,A.old,AND,0,2.51184,29.444444,360.165756,8.397086,1098,1.0,2.0,5.0,0.0,0.0,0.000000,0.0
44,B.new,AND,0,2.536919,30.773929,366.750228,8.385597,1097,1.0,2.0,5.0,0.009984,0.045152,0.018282,-0.001368
22,A.old,IOS,0,2.729091,29.416,396.502182,9.036,2750,1.0,2.0,6.0,0.0,0.0,0.000000,0.0
66,B.new,IOS,0,2.535409,29.132296,382.549157,8.69987,3855,1.0,2.0,5.0,-0.07097,-0.009645,-0.035190,-0.037199
1,A.old,AND,1,3.81746,38.678571,421.557540,6.97619,504,1.0,2.0,8.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,B.new,IOS,20,4.706468,38.686567,404.333333,3.776119,201,1.0,3.0,10.0,-0.0239,0.032602,-0.036946,0.082488
21,A.old,AND,21,4.672414,47.568966,492.206897,3.413793,58,1.0,3.0,11.3,0.0,0.0,0.000000,0.0
65,B.new,AND,21,4.044444,43.244444,398.600000,3.911111,45,1.0,3.0,8.0,-0.134399,-0.090911,-0.190178,0.145679
43,A.old,IOS,21,4.221374,32.633588,377.664122,2.961832,131,1.0,2.0,10.0,0.0,0.0,0.000000,0.0


### Sessions

In [62]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


In [63]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p10_sessions',
              color='FTUE_flag',
              title='10th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p50_sessions',
              color='FTUE_flag',
              title='Median sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p90_sessions',
              color='FTUE_flag',
              title='90th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


### Minutes

In [64]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Merges

In [65]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Taks

In [66]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

<!-- hide -->  
## Game day reached at day x (work in progress)

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [67]:
# hide-output
# Compute weighted average game day progression by install cohort and days since install

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-05-04,0,596,1.824194
1,2026-05-04,1,198,3.959108
2,2026-05-04,2,114,5.061321
3,2026-05-05,0,686,2.402542
4,2026-05-05,1,260,5.003040
...,...,...,...,...
122,2026-06-26,0,453,1.764706
123,2026-06-26,1,128,3.334975
124,2026-06-27,0,458,1.801282
125,2026-06-27,1,128,3.593301


In [68]:
# hide-output
# Line chart: weighted average game day by install cohort
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [69]:
# hide-output
# Compute and plot P10/P50/P90 game day distribution by days since install
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [70]:
# hide-output
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./earlyftue.ipynb',
    output_path='./earlyftue.html',
)

Saved to earlyftue.html


PosixPath('earlyftue.html')